
### Structure Output
Structured Output in LangChain means making an AI model return its response in a fixed and organized format, such as JSON, a dictionary, or a defined schema, instead of normal free-form text. It helps applications easily read and use the model's response. For example, instead of returning “The weather is sunny and 25°C”, structured output can return {"condition": "sunny", "temperature": 25}. In LangChain, structured output is commonly created using Pydantic models, TypedDict, or JSON Schema.


### Pydantic
Pydantic is a Python library used to validate and structure data using Python type hints.

In [17]:

import os 
from langchain.chat_models import  init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-20b")

model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001EBF26F4550>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001EBF26F4F50>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [2]:

from pydantic import BaseModel,Field

class Structure_model(BaseModel):
    Name:str = Field(description="Write the name of movie : ")
    Release_Date:int = Field(description="Write the date of movie realse : ")
    Director:str = Field(description="Write the movie directore name ")
    Rating:float = Field(description="write the rating of movie upto 10 ")


model_w_s = model.with_structured_output(Structure_model)

model_w_s


_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A662B12660>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A662B13380>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Structure_model', 'description': '', 'parameters': {'properties': {'Name': {'description':

In [11]:

# without structure 

respone = model.invoke("Tell me about the endgame movie ?")
print(respone)

content='**Avengers: Endgame** (2019) is the fourth and final installment of the Marvel Cinematic Universe’s “Avengers” saga, directed by Anthony and Joe Russo. It picks up directly after the catastrophic events of *Avengers: Infinity\u202fStone* (2018), where the villain Thanos wiped out half of all life with the snap of the Infinity Gauntlet.\n\n### Plot Overview\n- **Time‑Travel Mission**: The surviving heroes (Iron Man, Captain America, Thor, Hulk, Black Widow, Hawkeye, Ant‑Man, and others) devise a plan to travel back in time to retrieve the Infinity Stones before Thanos can use them.  \n- **Personal Journeys**: Each team member confronts their own grief and loss—Tony Stark’s redemption arc, Steve Rogers’s final farewell to a world he helped save, Thor’s struggle with his identity, etc.  \n- **The Final Battle**: After collecting the Stones, the Avengers assemble a massive army to confront Thanos and his forces in a climactic showdown. The film culminates in a dramatic sacrifice, 

In [8]:

## with the structure

respone = model_w_s.invoke("tell me the movie which get ratting out of 10 / 10?")
print(respone)

Name='The Godfather' Release_Date=1972 Director='Francis Ford Coppola' Rating=10.0



### Message Output according to parse structure 

In [ ]:

from pydantic import  BaseModel,Field

class movie(BaseModel):
    """ Details about the movie : """
    Name:str = Field(...,description="Write the name of movie")
    Release_Date:int = Field(...,description="Write the date of movie realse : ")
    Director:str = Field(...,description="Write the movie directore name ")
    Rating:float = Field(...,description="write the rating of movie upto 10 ")


ask = model.with_structured_output(movie,include_raw=True)
responed = ask.invoke("tell me about the movie goodfather ? ")
print(responed)

#  ai respond in structure format :
# include-raw mean show output exactly what ai produce


{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "tell me about the movie goodfather ?" They want info about the movie "Goodfather"? Probably they mean "The Godfather". We can provide info. The tool requires us to use the function to provide details. The function signature: movie({Director: string, Name: string, Rating: number, Release_Date: number}). The tool likely expects us to call it with appropriate values.\n\nWe should call the function with the data: Director: "Francis Ford Coppola", Name: "The Godfather", Rating: maybe 9.2, Release_Date: 1972. But Release_Date expects a number. Probably the year. So 1972. Or maybe 1972-03-24. But they expect number. We\'ll supply 1972. The rating is number up to 10. We\'ll give 9.2. Then we will call the function.\n\nWe need to use the tool.', 'tool_calls': [{'id': 'fc_b222cec4-4ebb-4931-b1d8-dc347278ae4e', 'function': {'arguments': '{"Director":"Francis Ford Coppola","Name":"The Godfather","Rating":9.2,"Re

### Structure Nested

In [4]:

## another example pydantic 

from pydantic import  BaseModel,Field

class hero_list(BaseModel):
    Name:str
    Role:str


class movie_details(BaseModel):
    Name:str
    Release_Date:str
    Caster:list[hero_list]
    Genre:list[str]
    Budget:float |None = Field(None,description="Budget in millions usd")


ask = model.with_structured_output(movie_details)
respone = ask.invoke("details about the inception")
print(respone)

Name='Inception' Release_Date='2010-07-16' Caster=[hero_list(Name='Leonardo DiCaprio', Role='Dom Cobb'), hero_list(Name='Joseph Gordon-Levitt', Role='Arthur'), hero_list(Name='Elliot Page', Role='Ariadne'), hero_list(Name='Tom Hardy', Role='Eames'), hero_list(Name='Ken Watanabe', Role='Saito'), hero_list(Name='Marion Cotillard', Role='Mal')] Genre=['Action', 'Adventure', 'Sci-Fi'] Budget=160000000.0



### TypeDict 

TypedDict in LangChain is a Python type used to define the structure and expected data types of a dictionary. It tells LangChain what fields the output should contain and what type of value each field should have.

In simple words: TypedDict acts like a template for dictionary data.

In [5]:

from typing_extensions import TypedDict,Annotated

class Movie(TypedDict):
    Name : Annotated[str,...,"write the name of movie : "]
    Release_Date : Annotated[int,...,"write the release Date of Movie : "]
    Director : Annotated[str,...,"Write name of director of movie : "]
    Budget : Annotated[float,...,"Bugdet in million Usd"]


model_typedict = model.with_structured_output(Movie)

respone = model_typedict.invoke("Tell me about the avengers movie : ")

print(respone)


{'Budget': 220, 'Director': 'Joss Whedon', 'Name': 'The Avengers', 'Release_Date': 2012}


In [7]:

## with nested structure


from typing_extensions import  TypedDict,Annotated

class Hero(TypedDict):
    Name:str
    Role:str

class movie_details(TypedDict):
    Name:str
    Release_Date:str
    Caster:list[Hero]
    Genre:list[str]
    Budget:Annotated[float,...,"Budget in millions usd"]


model_typedict = model.with_structured_output(movie_details)

respone = model_typedict.invoke("Tell me about the avengers movie : ")

respone



{'Budget': 220,
 'Caster': [{'Name': 'Robert Downey Jr.', 'Role': 'Tony Stark / Iron Man'},
  {'Name': 'Chris Evans', 'Role': 'Steve Rogers / Captain America'},
  {'Name': 'Mark Ruffalo', 'Role': 'Bruce Banner / Hulk'},
  {'Name': 'Chris Hemsworth', 'Role': 'Thor'},
  {'Name': 'Scarlett Johansson', 'Role': 'Natasha Romanoff / Black Widow'},
  {'Name': 'Jeremy Renner', 'Role': 'Clint Barton / Hawkeye'}],
 'Genre': ['Action', 'Adventure', 'Sci-Fi'],
 'Name': 'Avengers',
 'Release_Date': '2012-05-04'}

In [9]:

## profile of normal model

model.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}


### Data Class
Dataclass in LangChain: A dataclass is a Python class used to store structured data in a simple and organized way, with predefined fields and their data types.

In [19]:

import os

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


In [25]:

### with TypedDict

from typing_extensions import TypedDict
from langchain.agents import  create_agent

class Person(TypedDict):
    """contact information of person"""
    Name : str
    Gmail : str
    Contact : str

agent = create_agent(
    model = 'gpt-5',
    response_format = Person
)

respone =agent.invoke({"messages":"Extract the information of given : (john nepali,john@neplai.com,(455) 9874222)"})

respone

print(respone["structured_response"])

{'Name': 'john nepali', 'Gmail': 'john@neplai.com', 'Contact': '(455) 9874222'}


In [ ]:

###  with pydantic

from langchain.agents import  create_agent
from pydantic import BaseModel

class Movie(BaseModel):
    """ details about the Movie """
    Movie_name:str
    Main_Hero:str
    Director_name:str

agent = create_agent(
    model='gpt-5',
    response_format = Movie
)

respone =agent.invoke({"messages":"Extract the information of given : (Mypride,Dayahang Rai,Ram babu gurung"})



print(respone["structured_response"])


Movie_name='Mypride' Main_Hero='Dayahang Rai' Director_name='Ram babu gurung'


In [30]:

### Dataclass

from langchain.agents import  create_agent
from dataclasses import  dataclass

@dataclass
class Movie:
    Movie_name:str
    Hero_name :str
    Duration :str

agent = create_agent(
    model = "gpt-5",
    response_format = Movie
)

respone = agent.invoke({'messages':'extaract the information of given : (Mynepal,Saugat Malla, 2hrs)'})

print(respone["structured_response"])

Movie(Movie_name='Mynepal', Hero_name='Saugat Malla', Duration='2hrs')
